# Predicting Card Mana Value in the MTG Star Trek Set
## 1. Project Setup and Data Ingestion
This notebook executes the data pipeline and Random Forest classification model outlined in the PACE Strategy Document. The goal is to predict a card's balanced mana cost based on its base stats and unstructured ability text using Natural Language Processing (NLP).

In [5]:
import pandas as pd
import requests
import time

headers = {'User-Agent': 'MTG_Data_Project/1.0'}

# Search for the main set (trk) AND the supplementary/Commander set (trkc)
# Scryfall groups by unique card name by default, preventing duplicate alternate arts
url = "https://api.scryfall.com/cards/search?q=set:trk+OR+set:trkc"

all_cards = []
has_more = True

print("Fetching data from Scryfall...")

while has_more:
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        all_cards.extend(data['data']) # Add this page's cards to our master list
        
        has_more = data.get('has_more', False) # Check if there is another page
        if has_more:
            url = data['next_page'] # Overwrite the URL with the next page link
            time.sleep(0.1) # Best practice: pause 100ms between requests
    else:
        print(f"Error {response.status_code}: {response.json().get('details', 'Failed')}")
        break

print(f"Successfully downloaded {len(all_cards)} unique cards.")

Fetching data from Scryfall...
Successfully downloaded 311 unique cards.


## 2. Data Loading & Initial Exploration
With the raw JSON data downloaded, the next step is to load it into a structured Pandas DataFrame. This allows us to inspect the columns provided by the Scryfall API and identify which features (like mana cost, power, toughness, and oracle text) we will need to isolate for the Random Forest model.+

In [6]:
# Convert the list of JSON card dictionaries into a Pandas DataFrame
df = pd.DataFrame(all_cards)

# Display the shape (rows, columns)
print(f"Dataframe shape: {df.shape}")

# Display the first 5 rows to inspect the data structure
df.head()

Dataframe shape: (311, 71)


,object,id,oracle_id,multiverse_ids,mtgo_id,tcgplayer_id,cardmarket_id,name,lang,released_at,...,related_uris,purchase_uris,power,toughness,all_parts,preview,frame_effects,security_stamp,produced_mana,penny_rank
0,card,02080f42-863a-471b-992b-67e35ef1b7b7,9356aee1-3d2b-4d17-8863-663e1ddf56f3,[],158142,728620,917666.0,"70,000 Light-Years from Home",en,2026-11-13,...,{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,card,660ec8bc-dc03-4b42-84f3-091c19e2a853,8802b7f0-1ed1-4872-9939-182afb8441ca,[],158414,726700,919806.0,Academy Instructor,en,2026-11-13,...,{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,0,3,"[{'object': 'related_card', 'id': '660ec8bc-dc...",NaN,NaN,NaN,NaN,NaN
2,card,41644f06-d08f-406c-a56a-47e1b5273982,37f3eda9-3242-4d62-846f-935543d8f4fd,[],158504,725944,918938.0,Adversary by Algorithm,en,2026-11-13,...,{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,NaN,NaN,"[{'object': 'related_card', 'id': '41644f06-d0...","{'source': 'Beijing Premodern', 'source_uri': ...",NaN,NaN,NaN,NaN
3,card,e2b350b2-0eda-4c2b-b9d6-b58ab4848545,4c4559a5-a7a4-4a40-8a8f-e7375c741748,[],158362,729606,917744.0,A Good Day to Die,en,2026-11-13,...,{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,card,fe5dda31-159c-49c7-962e-a04b771256ea,61dc9770-1e29-49fa-a31b-b63b5f0496fb,[],158592,725820,918742.0,All Good Things...,en,2026-11-13,...,{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,NaN,NaN,NaN,"{'source': 'Polygon', 'source_uri': 'https://w...",[enchantment],oval,NaN,NaN
